# CA2FL: cached update calibration (K=5)

Buffered async FL in which the server **remembers the latest update of every client** and uses these cached updates to correct each global step. Based on Wang et al., *Tackling the Data Heterogeneity in Asynchronous Federated Learning with Cached Update Calibration* (ICLR 2024).

**How it works**

1. The server keeps a cached update `h_i` for every client (all zeros at the start) and their average `h`.
2. Clients train continuously; arriving updates wait in a buffer.
3. When the buffer is full: `v = h + average over the buffer of (delta_i - h_i)`, then `global += server_rate * v`.
4. Each client in the buffer then gets its new delta as its cache entry: `h_i = delta_i`. `h` is recomputed as the average over **all** clients.

> **Note:** `h` stands in for the clients that are **not** in the buffer, so every step includes an estimate of every client's contribution. Fast clients can't drag the model toward their own data, and slow, non-IID clients are still heard through their cached update. In the first global step all caches are zero, so it is the same as an unweighted FedBuff step.

> **Caution:** simplifications compared with the paper:
> * Updates are averaged **without** sample-count or staleness weights, as in the paper's Algorithm 2.
> * If a fast client appears twice in one buffer, both of its deltas are calibrated with the cache from the start of that step, and the average divides by the number of buffered updates.
> * The memory-saving quantised cache (MF-CA2FL) is not implemented.
> * The paper uses `server_rate` 1.0; here, with momentum and 2 local epochs, it oscillates (validation 88% vs 93.5% at 0.5).
> * By default **all** clients train at once, like the other approaches. Set `concurrency` to use the paper's random sampling of `M_c` active clients.

## 1. Load the simulator

Gives this notebook everything from `simulator.ipynb`: the settings, `Approach`, `fedavg`, `apply_delta` and `run`.

In [1]:
%run ../simulator.ipynb

## 2. The approach

Parameters are the class attributes at the top. Change them here, or pass them when running (see the tip below).

In [2]:
class CA2FL(Approach):
    name = "ca2fl"
    label = "CA2FL cached calibration K=5"          # shown in plot legends
    buffer_size = 5               # updates per global step (M in the paper)
    server_rate = 0.5             # global step size (eta in the paper); 0.5 scored best on validation here, as for FedBuff
    concurrency = None            # clients training at once (M_c); None = all clients, like the other approaches

    def start(self, server):
        zeros = {k: torch.zeros_like(v) for k, v in server.model.items()}
        self.cache = [zeros] * len(server.clients)          # h_i: latest update of each client
        self.h = zeros                                     # average of all h_i
        self.buffer = []
        if self.concurrency is None:
            super().start(server)
        else:
            for cid in server.rng.choice(len(server.clients), self.concurrency, replace=False):
                server.train(int(cid))

    def client_free(self, server, client):
        if self.concurrency is None:
            server.train(client.id)
        else:                                              # sample another idle client
            idle = [c.id for c in server.clients if not c.busy]
            server.train(int(server.rng.choice(idle)))

    def on_update(self, server, update):
        self.buffer.append(update)
        if len(self.buffer) < self.buffer_size:
            return
        new_model = {}
        for k, cur in server.model.items():
            if not torch.is_floating_point(cur):
                new_model[k] = cur.clone()
                continue
            calibration = sum(u.delta[k] - self.cache[u.client_id][k] for u in self.buffer) / len(self.buffer)
            new_model[k] = cur + self.server_rate * (self.h[k] + calibration)
        for u in self.buffer:                              # refresh the caches of the clients just heard
            self.cache[u.client_id] = u.delta
        self.h = {k: sum(c[k] for c in self.cache) / len(self.cache) for k in self.h}
        server.set_model(new_model, self.buffer)
        self.buffer = []

## 3. Run it

Runs every seed listed in `settings.yaml` and saves the results to `results/`. Then open **`compare.ipynb`** to see it next to the other approaches.

> **Tip:** to try a variant without editing the class, run e.g. `run(CA2FL(server_rate=1.0))`, or `run(CA2FL(concurrency=5))` for the paper's client sampling. It is saved as a separate run.

In [3]:
results = run(CA2FL())

ca2fl                        seed=42   val= 94.14%  acc= 95.00%  time=   560.2s  updates=  100  worst10%= 93.88%  wall=6.4s


## 4. Quick look

The final evaluation for each seed:

In [4]:
results.groupby("seed").tail(1)[["seed", "time", "updates", "accuracy", "loss", "worst10_client_acc", "mean_staleness"]].round(4)

,seed,time,updates,accuracy,loss,worst10_client_acc,mean_staleness
20,42,560.1644,100,0.95,0.1734,0.9388,2.8
